In [1]:
# ! pip install langchain langchain-openai

In [2]:
pip show langchain

Name: langchain
Version: 1.2.17
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /Users/kanavbansal/Desktop/Hands-on Labs (Evoke)/evoke_b1/13. Agents/9. MCP/3. Building MCP Server/.env/lib/python3.13/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [3]:
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY,
                 model="gpt-4o-mini",
                 temperature=0.0)

In [4]:
# ! pip install langchain-mcp-adapters

In [5]:
from langchain_mcp_adapters.client import MultiServerMCPClient

In [14]:
client = MultiServerMCPClient(
    {
        "maths-mcp": {
            "transport": "stdio",
            "command": "python",
            "args": [
                "mcp-server-stdio.py"
            ]
        }, 
        "weather-mcp": {
            "transport": "http",
            # Ensure you start your weather server on port 8000
            # Run your mcp server with command: "python mcp-server-http.py"
            "url": "http://localhost:8000/mcp",
        }
    }
)

In [15]:
time_toolset = await client.get_tools()

time_toolset

[StructuredTool(name='multiply', description='This tool takes two integers and returns the product', args_schema={'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x10ebc3e20>),
 StructuredTool(name='add', description='This tool takes two integers and returns the addition', args_schema={'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x10ebc3ec0>),
 StructuredTool(name='get_weather', description='Get weather for location.', args_schema={'additionalProperties': False, 'properties':

In [16]:
print(f"Loaded {len(time_toolset)} MCP Tools: {[tool.name for tool in time_toolset]}")

Loaded 3 MCP Tools: ['multiply', 'add', 'get_weather']


In [17]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=time_toolset
)

In [18]:
response = await agent.ainvoke({"messages": "What time is it?"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What time is it?
================================== Ai Message ==================================

I don't have real-time capabilities to check the current time. You can easily find the time by checking your device's clock or using a search engine.


In [19]:
response = await agent.ainvoke({"messages": "What is 123456 multiplied with 789012?"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What is 123456 multiplied with 789012?
================================== Ai Message ==================================
Tool Calls:
  multiply (call_k44t33dNWLmRgXT6xRiBthrk)
 Call ID: call_k44t33dNWLmRgXT6xRiBthrk
  Args:
    a: 123456
    b: 789012
================================= Tool Message =================================
Name: multiply

[{'type': 'text', 'text': '97408265472', 'id': 'lc_f6afb0dd-43aa-46d9-a0df-e4f9fe1c70b1'}]
================================== Ai Message ==================================

123456 multiplied by 789012 equals 97,408,265,472.


In [20]:
response = await agent.ainvoke({"messages": "What is the weather in Hyderabad?"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What is the weather in Hyderabad?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_AwPIQj19AsofBrwmhy0Kt4e6)
 Call ID: call_AwPIQj19AsofBrwmhy0Kt4e6
  Args:
    location: Hyderabad
================================= Tool Message =================================
Name: get_weather

[{'type': 'text', 'text': 'It is sunny in Hyderabad.', 'id': 'lc_174cc1cb-0c24-4988-a92a-92262354d499'}]
================================== Ai Message ==================================

The weather in Hyderabad is sunny.
